In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import HfFileSystem, hf_hub_download

In [ ]:
import torch
from autosim.utils import plot_spatiotemporal_video

In [ ]:
# 1. Connect to the Hugging Face filesystem
fs = HfFileSystem()
repo_id = "pdearena/NavierStokes-2D-conditoned" # Note the typo in their repo name

print(f"Listing files in {repo_id}...")

# 2. Find all .h5 simulation files in the repository
# The files are stored under the 'datasets/' prefix in the HfFileSystem
files = fs.ls(f"datasets/{repo_id}", detail=False)
h5_files = [f for f in files if f.endswith('.h5')]

if not h5_files:
    print("No .h5 files found!")
    # return
    
# We will just grab the very first trajectory to test
first_file_name = h5_files[0].split("/")[-1]

print(f"\nDownloading 1 trajectory: {first_file_name}")
print("This might take a minute depending on your connection...")

# 3. Download the file and cache it locally
file_path = hf_hub_download(
    repo_id=repo_id, 
    repo_type="dataset", 
    filename=first_file_name
)

print(f"\nDownload complete! File saved to cache at: {file_path}")


In [ ]:
def find_first_dataset(name, obj):
    """Callback to find the first actual dataset inside the HDF5 groups."""
    if isinstance(obj, h5py.Dataset):
        return obj
    return None

def print_h5_structure(name, obj):
    """Prints the name and shape of every dataset in the file."""
    if isinstance(obj, h5py.Dataset):
        print(f" - {name}: {obj.shape}")

In [ ]:
with h5py.File(file_path, 'r') as f:
    # 2. Map the short variable names to their full nested HDF5 paths
    channel_paths = {}
    def find_4d_arrays(name, obj):
        if isinstance(obj, h5py.Dataset) and len(obj.shape) == 4:
            key_name = name.split('/')[-1]
            channel_paths[key_name] = name
    f.visititems(find_4d_arrays)
    
    # 3. Setup our grid parameters
    target_keys = ['u', 'vx', 'vy']
    time_steps = [0, 8, 16, 24, 31]
    trajectory_idx = -1
    
    # Create a 5x3 grid of subplots
    fig, axes = plt.subplots(nrows=len(time_steps), ncols=len(target_keys), figsize=(12, 16))
    fig.suptitle(f"PDEArena data - Trajectory {trajectory_idx} Evolution", fontsize=18, y=0.98)
    
    for row_idx, t in enumerate(time_steps):
        for col_idx, key in enumerate(target_keys):
            ax = axes[row_idx, col_idx]
            full_path = channel_paths[key]
            
            # Extract the 2D spatial field for this specific time and channel
            field = f[full_path][trajectory_idx, t, :, :]
            
            # 4. Apply CFD-standard colormapping
            if key == 'u':
                # Buoyancy/Smoke is a positive scalar
                cmap = 'inferno'
                vmin, vmax = field.min(), field.max()
            else:
                # Velocities are vectors; center the colormap exactly at 0
                cmap = 'RdBu_r' 
                max_abs_val = max(abs(field.min()), abs(field.max()))
                vmin, vmax = -max_abs_val, max_abs_val
            
            im = ax.imshow(field, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
            
            # Clean up the axes ticks
            ax.set_xticks([])
            ax.set_yticks([])
            
            # Add Column Titles (Top Row Only)
            if row_idx == 0:
                ax.set_title(f"Channel: '{key}'", fontsize=16, fontweight='bold', pad=10)
                
            # Add Row Titles (Left Column Only)
            if col_idx == 0:
                ax.set_ylabel(f"Step {t}", fontsize=14, fontweight='bold', labelpad=10)
                
            # Add a colorbar to every individual subplot
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=9)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.94) # Make room for the suptitle
    plt.show()

In [ ]:
target_keys = ['u', 'vx', 'vy']
    
with h5py.File(file_path, 'r') as f:
    # Dynamically find the full paths for the keys
    channel_paths = {}
    f.visititems(lambda name, obj: channel_paths.update({name.split('/')[-1]: name}) 
                    if isinstance(obj, h5py.Dataset) and len(obj.shape) == 4 else None)
    
    # Load the 3 channels. Each has shape: (Batch, Time, X, Y)
    # We grab the first 5 trajectories to make a Batch of 5: shape (5, 56, 128, 128)
    c1 = f[channel_paths['u']][:5, :, :, :]
    c2 = f[channel_paths['vx']][:5, :, :, :]
    c3 = f[channel_paths['vy']][:5, :, :, :]
    
# Stack along the LAST axis to create the Channel dimension
# Shape becomes: (5, 56, 128, 128, 3) -> (Batch, Time, Width, Height, Channels)
stacked_numpy = np.stack([c1, c2, c3], axis=-1)

# Convert to PyTorch Tensor as expected by your type hints
true_tensor = torch.tensor(stacked_numpy, dtype=torch.float32)

print(f"Formatted Tensor Shape: {true_tensor.shape} (B, T, W, H, C)")

# 3. Call your autosim plotting function
print("Generating video...")
plot_spatiotemporal_video(
    true=true_tensor,
    pred=None,             # We just have the truth
    pred_uq=None,
    batch_idx=0,           # Plot the first simulation in the batch
    fps=10,
    cmap="magma",          # A good unified colormap for both buoyancy and velocities
    save_path="pdearena_autosim.mp4",
    title="PDEArena",
    colorbar_mode="column", # A clean mode from your function specs
    channel_names=['Buoyancy (u)', 'X-Velocity (vx)', 'Y-Velocity (vy)']
)